# 03 — Метрики и визуализации

Загружает данные и веса из предыдущих шагов.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from sklearn.calibration import calibration_curve

from liquefaction_ai import DEMO_PALETTE, load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.utils import (
    MODEL_DISPLAY_NAMES,
    collect_outputs,
    compute_metrics,
    localize_metric_table,
)
from liquefaction_ai.nn import DPIFlow, EVTNeuralSSM, GRUBaseline, RiskMLP, TCNBaseline

artifact_dir = Path("data/demo_run")
run_dir = Path("models/demo_run")

population, config = load_population_artifact(artifact_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark_data = prepare_benchmark_dataset(population, config, device)

test = benchmark_data["test"]
static_dim = test["static"].shape[1]
prefix_dim = test["prefix_summary"].shape[1]
seq_dim = test["seq_in"].shape[-1]

builders = {
    "MLP_risk": lambda: RiskMLP(static_dim=static_dim, prefix_dim=prefix_dim, hidden_dim=128),
    "GRU": lambda: GRUBaseline(static_dim=static_dim, seq_dim=seq_dim, hidden_dim=96),
    "TCN": lambda: TCNBaseline(static_dim=static_dim, seq_dim=seq_dim, hidden_dim=96),
    "DPI-Flow": lambda: DPIFlow(
        static_dim=static_dim,
        prefix_dim=prefix_dim,
        seq_len=config.seq_len,
        prefix_len=config.prefix_len,
        max_cycle_reference=config.max_cycle_reference,
        probabilistic=True,
        calibration_steps=2,
        use_analytical_layer=True,
    ),
    "EVT-NeuralSSM": lambda: EVTNeuralSSM(
        static_dim=static_dim,
        prefix_dim=prefix_dim,
        seq_dim=seq_dim,
        seq_len=config.seq_len,
        prefix_len=config.prefix_len,
        max_cycle_reference=config.max_cycle_reference,
    ),
}

file_key = {
    "MLP_risk": "mlp_risk",
    "GRU": "gru",
    "TCN": "tcn",
    "DPI-Flow": "dpi_flow",
    "EVT-NeuralSSM": "evt_ssm",
}

main_rows = []
sample_tables = {}

for display_name, factory in builders.items():
    key = file_key[display_name]
    model = factory().to(device)
    state_path = run_dir / f"{key}.pt"
    if not state_path.is_file():
        print("Пропуск (нет файла):", state_path)
        continue
    model.load_state_dict(torch.load(state_path, map_location=device, weights_only=True))
    outputs = collect_outputs(model, test, config, device)
    metrics, sample_df = compute_metrics(display_name, outputs, test, config)
    main_rows.append(metrics)
    sample_tables[display_name] = sample_df

leaderboard = pd.DataFrame(main_rows).sort_values(["Traj_RMSE", "Brier"], na_position="last")
display(localize_metric_table(leaderboard).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = leaderboard.dropna(subset=["Traj_RMSE"]).sort_values("Traj_RMSE")
axes[0].bar(plot_df["model"], plot_df["Traj_RMSE"], color=DEMO_PALETTE["primary"])
axes[0].set_title("RMSE траектории (тест)")
axes[0].tick_params(axis="x", rotation=25)

for name in plot_df["model"].head(4):
    st = sample_tables[name]
    frac_pos, mean_pred = calibration_curve(st["liq_label"], st["risk_prob_pred"], n_bins=10)
    axes[1].plot(mean_pred, frac_pos, marker="o", lw=2.0, label=MODEL_DISPLAY_NAMES.get(name, name))
axes[1].plot([0, 1], [0, 1], color=DEMO_PALETTE["dark"], ls="--", lw=1.2)
axes[1].set_title("Калибровка риска")
axes[1].legend()
fig.tight_layout()
plt.show()